In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('cairo')  # Use non-interactive backend for saving figures

# --- FILES ---
summary_path = "summaryAll2ndIteration.xlsx"      # species *_summary sheets; KEGG IDs in 'products'
class_path   = "CompoundClassification.xlsx"       # use 'Graphing' sheet (KEGG ID, Name, Classification)

# --- COLOR PALETTE (unchanged) ---
color_palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b",
    "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#393b79", "#8c6d31",
    "#e7ba52", "#e7969c", "#7b4173", "#a55194", "#843c39", "#ad494a",
    "#d6616b", "#e6550d", "#9c9ede", "#b5cf6b", "#6b6ecf", "#bd9e39",
    "#cedb9c", "#637939", "#8ca252", "#bcbddc", "#c7e9c0", "#e9a3c9"
]

# --- Load classification mapping from Graphing sheet (preserve casing) ---
graph_df = pd.read_excel(
    class_path, sheet_name="Graphing", usecols=["KEGG ID", "Name", "Classification"]
).dropna(subset=["KEGG ID", "Classification"])
graph_df["KEGG ID"] = graph_df["KEGG ID"].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=["KEGG ID"], keep="first")

# Build class list for colors (from mapping)
all_unique_classes = sorted(graph_df["Classification"].unique())
class_color_mapping = {cls: color_palette[i % len(color_palette)] for i, cls in enumerate(all_unique_classes)}

# --- Read summary workbook ---
excel_data = pd.ExcelFile(summary_path)
summary_sheet_names = [s for s in excel_data.sheet_names if s.endswith("_summary")]

# Helper to pick KEGG-ID column (prefers 'products')
def pick_id_col(df, hint="substrates"):
    if hint in df.columns:
        return hint
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\d{5}$").mean() > 0.5):
            return c
    return None

# Parse each species sheet → count KEGG IDs
all_substrate_frequencies = {}
for sheet in summary_sheet_names:
    species_name = sheet.replace("_summary", "")
    sheet_data = excel_data.parse(sheet)
    id_col = pick_id_col(sheet_data, "substrates")
    if id_col is None:
        continue
    substrate_counts = sheet_data[id_col].dropna().astype(str).str.strip().value_counts()
    all_substrate_frequencies[species_name] = pd.DataFrame({
        "Substrate": substrate_counts.index,
        "Frequency": substrate_counts.values
    })

# Combine species into one DataFrame
combined_substrate_frequencies = pd.concat(all_substrate_frequencies, names=["Species"]).reset_index(level=0)

# Merge with mapping to add Class (exact casing from Excel)
combined_substrate_with_class = combined_substrate_frequencies.merge(
    graph_df[["KEGG ID", "Classification"]],
    left_on="Substrate",
    right_on="KEGG ID",
    how="left"
)

# Drop unmapped and (optionally) exclude 'Free Radicals' while keeping other casings intact
combined_substrate_with_class = combined_substrate_with_class.dropna(subset=["Classification"])
combined_substrate_with_class = combined_substrate_with_class[
    combined_substrate_with_class["Classification"].astype(str).str.strip().str.lower() != "free radicals"
]

# Pivot: species × class → summed frequencies
class_by_species = combined_substrate_with_class.pivot_table(
    index="Species", columns="Classification", values="Frequency", aggfunc="sum", fill_value=0
)

# Sort species by total frequency (desc)
class_by_species["Total Frequency"] = class_by_species.sum(axis=1)
class_by_species = class_by_species.sort_values(by="Total Frequency", ascending=False).drop(columns="Total Frequency")

# Colors only for classes present in the pivot
colors = [class_color_mapping.get(cls, "#cccccc") for cls in class_by_species.columns]

# --- Plot stacked bar chart ---
plt.figure(figsize=(14, 8))
class_by_species.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 8),
    width=0.8,
    color=colors
)

plt.title("Substrate Class Utilization across Species")
plt.xlabel("Species")
plt.ylabel("Frequency of Utilization")

# Legend using exact class names from the data order
handles = [plt.Line2D([0], [0], color=class_color_mapping.get(cls, "#cccccc"), lw=4)
           for cls in class_by_species.columns]
plt.legend(handles, class_by_species.columns, title="Substrate Class",
           bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.savefig("UptakeUtilization2ndIterUnion.pdf", format="pdf", bbox_inches="tight")
plt.show()

# --- TABLE FIGURE (separate page) — INVERTED (classes as rows, species as columns) ---
tbl_df = class_by_species.T  # invert

fig_w = max(12, 2 + 0.35 * tbl_df.shape[1])   # width scales with #species (columns)
fig_h = max(6,  1.2 + 0.28 * tbl_df.shape[0]) # height scales with #classes (rows)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis('off')

tbl = ax.table(
    cellText=tbl_df.astype(int).values,   # remove .astype(int) if you want raw values
    rowLabels=tbl_df.index,
    colLabels=tbl_df.columns,
    cellLoc='center',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.25)  # widen cells / increase row height a bit

plt.tight_layout()
plt.savefig("UptakeUtilizationTable2ndIterUnion.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('cairo')  # Use non-interactive backend for saving figures

# --- FILES ---
summary_path = "summaryAll1stIteration.xlsx"      # species *_summary sheets; KEGG IDs in 'products'
class_path   = "CompoundClassification.xlsx"       # use 'Graphing' sheet (KEGG ID, Name, Classification)

# --- COLOR PALETTE (unchanged) ---
color_palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b",
    "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#393b79", "#8c6d31",
    "#e7ba52", "#e7969c", "#7b4173", "#a55194", "#843c39", "#ad494a",
    "#d6616b", "#e6550d", "#9c9ede", "#b5cf6b", "#6b6ecf", "#bd9e39",
    "#cedb9c", "#637939", "#8ca252", "#bcbddc", "#c7e9c0", "#e9a3c9"
]

# --- Load classification mapping from Graphing sheet (preserve casing) ---
graph_df = pd.read_excel(
    class_path, sheet_name="Graphing", usecols=["KEGG ID", "Name", "Classification"]
).dropna(subset=["KEGG ID", "Classification"])
graph_df["KEGG ID"] = graph_df["KEGG ID"].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=["KEGG ID"], keep="first")

# Build class list for colors (from mapping)
all_unique_classes = sorted(graph_df["Classification"].unique())
class_color_mapping = {cls: color_palette[i % len(color_palette)] for i, cls in enumerate(all_unique_classes)}

# --- Read summary workbook ---
excel_data = pd.ExcelFile(summary_path)
summary_sheet_names = [s for s in excel_data.sheet_names if s.endswith("_summary")]

# Helper to pick KEGG-ID column (prefers 'products')
def pick_id_col(df, hint="substrates"):
    if hint in df.columns:
        return hint
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\d{5}$").mean() > 0.5):
            return c
    return None

# Parse each species sheet → count KEGG IDs
all_substrate_frequencies = {}
for sheet in summary_sheet_names:
    species_name = sheet.replace("_summary", "")
    sheet_data = excel_data.parse(sheet)
    id_col = pick_id_col(sheet_data, "substrates")
    if id_col is None:
        continue
    substrate_counts = sheet_data[id_col].dropna().astype(str).str.strip().value_counts()
    all_substrate_frequencies[species_name] = pd.DataFrame({
        "Substrate": substrate_counts.index,
        "Frequency": substrate_counts.values
    })

# Combine species into one DataFrame
combined_substrate_frequencies = pd.concat(all_substrate_frequencies, names=["Species"]).reset_index(level=0)

# Merge with mapping to add Class (exact casing from Excel)
combined_substrate_with_class = combined_substrate_frequencies.merge(
    graph_df[["KEGG ID", "Classification"]],
    left_on="Substrate",
    right_on="KEGG ID",
    how="left"
)

# Drop unmapped and (optionally) exclude 'Free Radicals' while keeping other casings intact
combined_substrate_with_class = combined_substrate_with_class.dropna(subset=["Classification"])
combined_substrate_with_class = combined_substrate_with_class[
    combined_substrate_with_class["Classification"].astype(str).str.strip().str.lower() != "free radicals"
]

# Pivot: species × class → summed frequencies
class_by_species = combined_substrate_with_class.pivot_table(
    index="Species", columns="Classification", values="Frequency", aggfunc="sum", fill_value=0
)

# Sort species by total frequency (desc)
class_by_species["Total Frequency"] = class_by_species.sum(axis=1)
class_by_species = class_by_species.sort_values(by="Total Frequency", ascending=False).drop(columns="Total Frequency")

# Colors only for classes present in the pivot
colors = [class_color_mapping.get(cls, "#cccccc") for cls in class_by_species.columns]

# --- Plot stacked bar chart ---
plt.figure(figsize=(14, 8))
class_by_species.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 8),
    width=0.8,
    color=colors
)

plt.title("Substrate Class Utilization across Species")
plt.xlabel("Species")
plt.ylabel("Frequency of Utilization")

# Legend using exact class names from the data order
handles = [plt.Line2D([0], [0], color=class_color_mapping.get(cls, "#cccccc"), lw=4)
           for cls in class_by_species.columns]
plt.legend(handles, class_by_species.columns, title="Substrate Class",
           bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.savefig("UptakeUtilization1stIterUnion.pdf", format="pdf", bbox_inches="tight")
plt.show()

# --- TABLE FIGURE (separate page) — INVERTED (classes as rows, species as columns) ---
tbl_df = class_by_species.T  # invert

fig_w = max(12, 2 + 0.35 * tbl_df.shape[1])   # width scales with #species (columns)
fig_h = max(6,  1.2 + 0.28 * tbl_df.shape[0]) # height scales with #classes (rows)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis('off')

tbl = ax.table(
    cellText=tbl_df.astype(int).values,   # remove .astype(int) if you want raw values
    rowLabels=tbl_df.index,
    colLabels=tbl_df.columns,
    cellLoc='center',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.25)  # widen cells / increase row height a bit

plt.tight_layout()
plt.savefig("UptakeUtilizationTable1stIterUnion.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('cairo')  # Use non-interactive backend for saving figures

# --- FILES ---
summary_path = "summaryAll100_1stIteration.xlsx"      # species *_summary sheets; KEGG IDs in 'products'
class_path   = "CompoundClassification.xlsx"       # use 'Graphing' sheet (KEGG ID, Name, Classification)

# --- COLOR PALETTE (unchanged) ---
color_palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b",
    "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#393b79", "#8c6d31",
    "#e7ba52", "#e7969c", "#7b4173", "#a55194", "#843c39", "#ad494a",
    "#d6616b", "#e6550d", "#9c9ede", "#b5cf6b", "#6b6ecf", "#bd9e39",
    "#cedb9c", "#637939", "#8ca252", "#bcbddc", "#c7e9c0", "#e9a3c9"
]

# --- Load classification mapping from Graphing sheet (preserve casing) ---
graph_df = pd.read_excel(
    class_path, sheet_name="Graphing", usecols=["KEGG ID", "Name", "Classification"]
).dropna(subset=["KEGG ID", "Classification"])
graph_df["KEGG ID"] = graph_df["KEGG ID"].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=["KEGG ID"], keep="first")

# Build class list for colors (from mapping)
all_unique_classes = sorted(graph_df["Classification"].unique())
class_color_mapping = {cls: color_palette[i % len(color_palette)] for i, cls in enumerate(all_unique_classes)}

# --- Read summary workbook ---
excel_data = pd.ExcelFile(summary_path)
summary_sheet_names = [s for s in excel_data.sheet_names if s.endswith("_summary")]

# Helper to pick KEGG-ID column (prefers 'products')
def pick_id_col(df, hint="substrates"):
    if hint in df.columns:
        return hint
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\d{5}$").mean() > 0.5):
            return c
    return None

# Parse each species sheet → count KEGG IDs
all_substrate_frequencies = {}
for sheet in summary_sheet_names:
    species_name = sheet.replace("_summary", "")
    sheet_data = excel_data.parse(sheet)
    id_col = pick_id_col(sheet_data, "substrates")
    if id_col is None:
        continue
    substrate_counts = sheet_data[id_col].dropna().astype(str).str.strip().value_counts()
    all_substrate_frequencies[species_name] = pd.DataFrame({
        "Substrate": substrate_counts.index,
        "Frequency": substrate_counts.values
    })

# Combine species into one DataFrame
combined_substrate_frequencies = pd.concat(all_substrate_frequencies, names=["Species"]).reset_index(level=0)

# Merge with mapping to add Class (exact casing from Excel)
combined_substrate_with_class = combined_substrate_frequencies.merge(
    graph_df[["KEGG ID", "Classification"]],
    left_on="Substrate",
    right_on="KEGG ID",
    how="left"
)

# Drop unmapped and (optionally) exclude 'Free Radicals' while keeping other casings intact
combined_substrate_with_class = combined_substrate_with_class.dropna(subset=["Classification"])
combined_substrate_with_class = combined_substrate_with_class[
    combined_substrate_with_class["Classification"].astype(str).str.strip().str.lower() != "free radicals"
]

# Pivot: species × class → summed frequencies
class_by_species = combined_substrate_with_class.pivot_table(
    index="Species", columns="Classification", values="Frequency", aggfunc="sum", fill_value=0
)

# Sort species by total frequency (desc)
class_by_species["Total Frequency"] = class_by_species.sum(axis=1)
class_by_species = class_by_species.sort_values(by="Total Frequency", ascending=False).drop(columns="Total Frequency")

# Colors only for classes present in the pivot
colors = [class_color_mapping.get(cls, "#cccccc") for cls in class_by_species.columns]

# --- Plot stacked bar chart ---
plt.figure(figsize=(14, 8))
class_by_species.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 8),
    width=0.8,
    color=colors
)

plt.title("Substrate Class Utilization across Species")
plt.xlabel("Species")
plt.ylabel("Frequency of Utilization")

# Legend using exact class names from the data order
handles = [plt.Line2D([0], [0], color=class_color_mapping.get(cls, "#cccccc"), lw=4)
           for cls in class_by_species.columns]
plt.legend(handles, class_by_species.columns, title="Substrate Class",
           bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.savefig("UptakeUtilization1stIter100.pdf", format="pdf", bbox_inches="tight")
plt.show()

# --- TABLE FIGURE (separate page) — INVERTED (classes as rows, species as columns) ---
tbl_df = class_by_species.T  # invert

fig_w = max(12, 2 + 0.35 * tbl_df.shape[1])   # width scales with #species (columns)
fig_h = max(6,  1.2 + 0.28 * tbl_df.shape[0]) # height scales with #classes (rows)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis('off')

tbl = ax.table(
    cellText=tbl_df.astype(int).values,   # remove .astype(int) if you want raw values
    rowLabels=tbl_df.index,
    colLabels=tbl_df.columns,
    cellLoc='center',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.25)  # widen cells / increase row height a bit

plt.tight_layout()
plt.savefig("UptakeUtilizationTable1stIter100.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('cairo')  # Use non-interactive backend for saving figures

# --- FILES ---
summary_path = "summaryAll100_2ndIteration.xlsx"      # species *_summary sheets; KEGG IDs in 'products'
class_path   = "CompoundClassification.xlsx"       # use 'Graphing' sheet (KEGG ID, Name, Classification)

# --- COLOR PALETTE (unchanged) ---
color_palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b",
    "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#393b79", "#8c6d31",
    "#e7ba52", "#e7969c", "#7b4173", "#a55194", "#843c39", "#ad494a",
    "#d6616b", "#e6550d", "#9c9ede", "#b5cf6b", "#6b6ecf", "#bd9e39",
    "#cedb9c", "#637939", "#8ca252", "#bcbddc", "#c7e9c0", "#e9a3c9"
]

# --- Load classification mapping from Graphing sheet (preserve casing) ---
graph_df = pd.read_excel(
    class_path, sheet_name="Graphing", usecols=["KEGG ID", "Name", "Classification"]
).dropna(subset=["KEGG ID", "Classification"])
graph_df["KEGG ID"] = graph_df["KEGG ID"].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=["KEGG ID"], keep="first")

# Build class list for colors (from mapping)
all_unique_classes = sorted(graph_df["Classification"].unique())
class_color_mapping = {cls: color_palette[i % len(color_palette)] for i, cls in enumerate(all_unique_classes)}

# --- Read summary workbook ---
excel_data = pd.ExcelFile(summary_path)
summary_sheet_names = [s for s in excel_data.sheet_names if s.endswith("_summary")]

# Helper to pick KEGG-ID column (prefers 'products')
def pick_id_col(df, hint="substrates"):
    if hint in df.columns:
        return hint
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\d{5}$").mean() > 0.5):
            return c
    return None

# Parse each species sheet → count KEGG IDs
all_substrate_frequencies = {}
for sheet in summary_sheet_names:
    species_name = sheet.replace("_summary", "")
    sheet_data = excel_data.parse(sheet)
    id_col = pick_id_col(sheet_data, "substrates")
    if id_col is None:
        continue
    substrate_counts = sheet_data[id_col].dropna().astype(str).str.strip().value_counts()
    all_substrate_frequencies[species_name] = pd.DataFrame({
        "Substrate": substrate_counts.index,
        "Frequency": substrate_counts.values
    })

# Combine species into one DataFrame
combined_substrate_frequencies = pd.concat(all_substrate_frequencies, names=["Species"]).reset_index(level=0)

# Merge with mapping to add Class (exact casing from Excel)
combined_substrate_with_class = combined_substrate_frequencies.merge(
    graph_df[["KEGG ID", "Classification"]],
    left_on="Substrate",
    right_on="KEGG ID",
    how="left"
)

# Drop unmapped and (optionally) exclude 'Free Radicals' while keeping other casings intact
combined_substrate_with_class = combined_substrate_with_class.dropna(subset=["Classification"])
combined_substrate_with_class = combined_substrate_with_class[
    combined_substrate_with_class["Classification"].astype(str).str.strip().str.lower() != "free radicals"
]

# Pivot: species × class → summed frequencies
class_by_species = combined_substrate_with_class.pivot_table(
    index="Species", columns="Classification", values="Frequency", aggfunc="sum", fill_value=0
)

# Sort species by total frequency (desc)
class_by_species["Total Frequency"] = class_by_species.sum(axis=1)
class_by_species = class_by_species.sort_values(by="Total Frequency", ascending=False).drop(columns="Total Frequency")

# Colors only for classes present in the pivot
colors = [class_color_mapping.get(cls, "#cccccc") for cls in class_by_species.columns]

# --- Plot stacked bar chart ---
plt.figure(figsize=(14, 8))
class_by_species.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 8),
    width=0.8,
    color=colors
)

plt.title("Substrate Class Utilization across Species")
plt.xlabel("Species")
plt.ylabel("Frequency of Utilization")

# Legend using exact class names from the data order
handles = [plt.Line2D([0], [0], color=class_color_mapping.get(cls, "#cccccc"), lw=4)
           for cls in class_by_species.columns]
plt.legend(handles, class_by_species.columns, title="Substrate Class",
           bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.savefig("UptakeUtilization2ndIter100.pdf", format="pdf", bbox_inches="tight")
plt.show()

# --- TABLE FIGURE (separate page) — INVERTED (classes as rows, species as columns) ---
tbl_df = class_by_species.T  # invert

fig_w = max(12, 2 + 0.35 * tbl_df.shape[1])   # width scales with #species (columns)
fig_h = max(6,  1.2 + 0.28 * tbl_df.shape[0]) # height scales with #classes (rows)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis('off')

tbl = ax.table(
    cellText=tbl_df.astype(int).values,   # remove .astype(int) if you want raw values
    rowLabels=tbl_df.index,
    colLabels=tbl_df.columns,
    cellLoc='center',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.25)  # widen cells / increase row height a bit

plt.tight_layout()
plt.savefig("UptakeUtilizationTable2ndIter100.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('cairo')  # Use non-interactive backend for saving figures

# --- FILES ---
summary_path = "summaryAll100_2ndIteration.xlsx"      # species *_summary sheets; KEGG IDs in 'products'
class_path   = "CompoundClassification.xlsx"       # use 'Graphing' sheet (KEGG ID, Name, Classification)

# --- COLOR PALETTE (unchanged) ---
color_palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b",
    "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#393b79", "#8c6d31",
    "#e7ba52", "#e7969c", "#7b4173", "#a55194", "#843c39", "#ad494a",
    "#d6616b", "#e6550d", "#9c9ede", "#b5cf6b", "#6b6ecf", "#bd9e39",
    "#cedb9c", "#637939", "#8ca252", "#bcbddc", "#c7e9c0", "#e9a3c9"
]

# --- Load classification mapping from Graphing sheet (preserve casing) ---
graph_df = pd.read_excel(
    class_path, sheet_name="Graphing", usecols=["KEGG ID", "Name", "Classification"]
).dropna(subset=["KEGG ID", "Classification"])
graph_df["KEGG ID"] = graph_df["KEGG ID"].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=["KEGG ID"], keep="first")

# Build class list for colors (from mapping)
all_unique_classes = sorted(graph_df["Classification"].unique())
class_color_mapping = {cls: color_palette[i % len(color_palette)] for i, cls in enumerate(all_unique_classes)}

# --- Read summary workbook ---
excel_data = pd.ExcelFile(summary_path)
summary_sheet_names = [s for s in excel_data.sheet_names if s.endswith("_summary")]

# Helper to pick KEGG-ID column (prefers 'products')
def pick_id_col(df, hint="products"):
    if hint in df.columns:
        return hint
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\d{5}$").mean() > 0.5):
            return c
    return None

# Parse each species sheet → count KEGG IDs
all_substrate_frequencies = {}
for sheet in summary_sheet_names:
    species_name = sheet.replace("_summary", "")
    sheet_data = excel_data.parse(sheet)
    id_col = pick_id_col(sheet_data, "products")
    if id_col is None:
        continue
    substrate_counts = sheet_data[id_col].dropna().astype(str).str.strip().value_counts()
    all_substrate_frequencies[species_name] = pd.DataFrame({
        "Substrate": substrate_counts.index,
        "Frequency": substrate_counts.values
    })

# Combine species into one DataFrame
combined_substrate_frequencies = pd.concat(all_substrate_frequencies, names=["Species"]).reset_index(level=0)

# Merge with mapping to add Class (exact casing from Excel)
combined_substrate_with_class = combined_substrate_frequencies.merge(
    graph_df[["KEGG ID", "Classification"]],
    left_on="Substrate",
    right_on="KEGG ID",
    how="left"
)

# Drop unmapped and (optionally) exclude 'Free Radicals' while keeping other casings intact
combined_substrate_with_class = combined_substrate_with_class.dropna(subset=["Classification"])
combined_substrate_with_class = combined_substrate_with_class[
    combined_substrate_with_class["Classification"].astype(str).str.strip().str.lower() != "free radicals"
]

# Pivot: species × class → summed frequencies
class_by_species = combined_substrate_with_class.pivot_table(
    index="Species", columns="Classification", values="Frequency", aggfunc="sum", fill_value=0
)

# Sort species by total frequency (desc)
class_by_species["Total Frequency"] = class_by_species.sum(axis=1)
class_by_species = class_by_species.sort_values(by="Total Frequency", ascending=False).drop(columns="Total Frequency")

# Colors only for classes present in the pivot
colors = [class_color_mapping.get(cls, "#cccccc") for cls in class_by_species.columns]

# --- Plot stacked bar chart ---
plt.figure(figsize=(14, 8))
class_by_species.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 8),
    width=0.8,
    color=colors
)

plt.title("Substrate Class Utilization across Species")
plt.xlabel("Species")
plt.ylabel("Frequency of Utilization")

# Legend using exact class names from the data order
handles = [plt.Line2D([0], [0], color=class_color_mapping.get(cls, "#cccccc"), lw=4)
           for cls in class_by_species.columns]
plt.legend(handles, class_by_species.columns, title="Substrate Class",
           bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.savefig("SecretionUtilization2ndIter100.pdf", format="pdf", bbox_inches="tight")
plt.show()

# --- TABLE FIGURE (separate page) — INVERTED (classes as rows, species as columns) ---
tbl_df = class_by_species.T  # invert

fig_w = max(12, 2 + 0.35 * tbl_df.shape[1])   # width scales with #species (columns)
fig_h = max(6,  1.2 + 0.28 * tbl_df.shape[0]) # height scales with #classes (rows)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis('off')

tbl = ax.table(
    cellText=tbl_df.astype(int).values,   # remove .astype(int) if you want raw values
    rowLabels=tbl_df.index,
    colLabels=tbl_df.columns,
    cellLoc='center',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.25)  # widen cells / increase row height a bit

plt.tight_layout()
plt.savefig("SecretionUtilizationTable2ndIter100.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('cairo')  # Use non-interactive backend for saving figures

# --- FILES ---
summary_path = "summaryAll100_1stIteration.xlsx"      # species *_summary sheets; KEGG IDs in 'products'
class_path   = "CompoundClassification.xlsx"       # use 'Graphing' sheet (KEGG ID, Name, Classification)

# --- COLOR PALETTE (unchanged) ---
color_palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b",
    "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#393b79", "#8c6d31",
    "#e7ba52", "#e7969c", "#7b4173", "#a55194", "#843c39", "#ad494a",
    "#d6616b", "#e6550d", "#9c9ede", "#b5cf6b", "#6b6ecf", "#bd9e39",
    "#cedb9c", "#637939", "#8ca252", "#bcbddc", "#c7e9c0", "#e9a3c9"
]

# --- Load classification mapping from Graphing sheet (preserve casing) ---
graph_df = pd.read_excel(
    class_path, sheet_name="Graphing", usecols=["KEGG ID", "Name", "Classification"]
).dropna(subset=["KEGG ID", "Classification"])
graph_df["KEGG ID"] = graph_df["KEGG ID"].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=["KEGG ID"], keep="first")

# Build class list for colors (from mapping)
all_unique_classes = sorted(graph_df["Classification"].unique())
class_color_mapping = {cls: color_palette[i % len(color_palette)] for i, cls in enumerate(all_unique_classes)}

# --- Read summary workbook ---
excel_data = pd.ExcelFile(summary_path)
summary_sheet_names = [s for s in excel_data.sheet_names if s.endswith("_summary")]

# Helper to pick KEGG-ID column (prefers 'products')
def pick_id_col(df, hint="products"):
    if hint in df.columns:
        return hint
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\d{5}$").mean() > 0.5):
            return c
    return None

# Parse each species sheet → count KEGG IDs
all_substrate_frequencies = {}
for sheet in summary_sheet_names:
    species_name = sheet.replace("_summary", "")
    sheet_data = excel_data.parse(sheet)
    id_col = pick_id_col(sheet_data, "products")
    if id_col is None:
        continue
    substrate_counts = sheet_data[id_col].dropna().astype(str).str.strip().value_counts()
    all_substrate_frequencies[species_name] = pd.DataFrame({
        "Substrate": substrate_counts.index,
        "Frequency": substrate_counts.values
    })

# Combine species into one DataFrame
combined_substrate_frequencies = pd.concat(all_substrate_frequencies, names=["Species"]).reset_index(level=0)

# Merge with mapping to add Class (exact casing from Excel)
combined_substrate_with_class = combined_substrate_frequencies.merge(
    graph_df[["KEGG ID", "Classification"]],
    left_on="Substrate",
    right_on="KEGG ID",
    how="left"
)

# Drop unmapped and (optionally) exclude 'Free Radicals' while keeping other casings intact
combined_substrate_with_class = combined_substrate_with_class.dropna(subset=["Classification"])
combined_substrate_with_class = combined_substrate_with_class[
    combined_substrate_with_class["Classification"].astype(str).str.strip().str.lower() != "free radicals"
]

# Pivot: species × class → summed frequencies
class_by_species = combined_substrate_with_class.pivot_table(
    index="Species", columns="Classification", values="Frequency", aggfunc="sum", fill_value=0
)

# Sort species by total frequency (desc)
class_by_species["Total Frequency"] = class_by_species.sum(axis=1)
class_by_species = class_by_species.sort_values(by="Total Frequency", ascending=False).drop(columns="Total Frequency")

# Colors only for classes present in the pivot
colors = [class_color_mapping.get(cls, "#cccccc") for cls in class_by_species.columns]

# --- Plot stacked bar chart ---
plt.figure(figsize=(14, 8))
class_by_species.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 8),
    width=0.8,
    color=colors
)

plt.title("Substrate Class Utilization across Species")
plt.xlabel("Species")
plt.ylabel("Frequency of Utilization")

# Legend using exact class names from the data order
handles = [plt.Line2D([0], [0], color=class_color_mapping.get(cls, "#cccccc"), lw=4)
           for cls in class_by_species.columns]
plt.legend(handles, class_by_species.columns, title="Substrate Class",
           bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.savefig("SecretionUtilization1stIter100.pdf", format="pdf", bbox_inches="tight")
plt.show()

# --- TABLE FIGURE (separate page) — INVERTED (classes as rows, species as columns) ---
tbl_df = class_by_species.T  # invert

fig_w = max(12, 2 + 0.35 * tbl_df.shape[1])   # width scales with #species (columns)
fig_h = max(6,  1.2 + 0.28 * tbl_df.shape[0]) # height scales with #classes (rows)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis('off')

tbl = ax.table(
    cellText=tbl_df.astype(int).values,   # remove .astype(int) if you want raw values
    rowLabels=tbl_df.index,
    colLabels=tbl_df.columns,
    cellLoc='center',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.25)  # widen cells / increase row height a bit

plt.tight_layout()
plt.savefig("SecretionUtilizationTable1stIter100.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('cairo')  # Use non-interactive backend for saving figures

# --- FILES ---
summary_path = "summaryAll1stIteration.xlsx"      # species *_summary sheets; KEGG IDs in 'products'
class_path   = "CompoundClassification.xlsx"       # use 'Graphing' sheet (KEGG ID, Name, Classification)

# --- COLOR PALETTE (unchanged) ---
color_palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b",
    "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#393b79", "#8c6d31",
    "#e7ba52", "#e7969c", "#7b4173", "#a55194", "#843c39", "#ad494a",
    "#d6616b", "#e6550d", "#9c9ede", "#b5cf6b", "#6b6ecf", "#bd9e39",
    "#cedb9c", "#637939", "#8ca252", "#bcbddc", "#c7e9c0", "#e9a3c9"
]

# --- Load classification mapping from Graphing sheet (preserve casing) ---
graph_df = pd.read_excel(
    class_path, sheet_name="Graphing", usecols=["KEGG ID", "Name", "Classification"]
).dropna(subset=["KEGG ID", "Classification"])
graph_df["KEGG ID"] = graph_df["KEGG ID"].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=["KEGG ID"], keep="first")

# Build class list for colors (from mapping)
all_unique_classes = sorted(graph_df["Classification"].unique())
class_color_mapping = {cls: color_palette[i % len(color_palette)] for i, cls in enumerate(all_unique_classes)}

# --- Read summary workbook ---
excel_data = pd.ExcelFile(summary_path)
summary_sheet_names = [s for s in excel_data.sheet_names if s.endswith("_summary")]

# Helper to pick KEGG-ID column (prefers 'products')
def pick_id_col(df, hint="products"):
    if hint in df.columns:
        return hint
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\d{5}$").mean() > 0.5):
            return c
    return None

# Parse each species sheet → count KEGG IDs
all_substrate_frequencies = {}
for sheet in summary_sheet_names:
    species_name = sheet.replace("_summary", "")
    sheet_data = excel_data.parse(sheet)
    id_col = pick_id_col(sheet_data, "products")
    if id_col is None:
        continue
    substrate_counts = sheet_data[id_col].dropna().astype(str).str.strip().value_counts()
    all_substrate_frequencies[species_name] = pd.DataFrame({
        "Substrate": substrate_counts.index,
        "Frequency": substrate_counts.values
    })

# Combine species into one DataFrame
combined_substrate_frequencies = pd.concat(all_substrate_frequencies, names=["Species"]).reset_index(level=0)

# Merge with mapping to add Class (exact casing from Excel)
combined_substrate_with_class = combined_substrate_frequencies.merge(
    graph_df[["KEGG ID", "Classification"]],
    left_on="Substrate",
    right_on="KEGG ID",
    how="left"
)

# Drop unmapped and (optionally) exclude 'Free Radicals' while keeping other casings intact
combined_substrate_with_class = combined_substrate_with_class.dropna(subset=["Classification"])
combined_substrate_with_class = combined_substrate_with_class[
    combined_substrate_with_class["Classification"].astype(str).str.strip().str.lower() != "free radicals"
]

# Pivot: species × class → summed frequencies
class_by_species = combined_substrate_with_class.pivot_table(
    index="Species", columns="Classification", values="Frequency", aggfunc="sum", fill_value=0
)

# Sort species by total frequency (desc)
class_by_species["Total Frequency"] = class_by_species.sum(axis=1)
class_by_species = class_by_species.sort_values(by="Total Frequency", ascending=False).drop(columns="Total Frequency")

# Colors only for classes present in the pivot
colors = [class_color_mapping.get(cls, "#cccccc") for cls in class_by_species.columns]

# --- Plot stacked bar chart ---
plt.figure(figsize=(14, 8))
class_by_species.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 8),
    width=0.8,
    color=colors
)

plt.title("Substrate Class Utilization across Species")
plt.xlabel("Species")
plt.ylabel("Frequency of Utilization")

# Legend using exact class names from the data order
handles = [plt.Line2D([0], [0], color=class_color_mapping.get(cls, "#cccccc"), lw=4)
           for cls in class_by_species.columns]
plt.legend(handles, class_by_species.columns, title="Substrate Class",
           bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.savefig("SecretionUtilization1stIterUnion.pdf", format="pdf", bbox_inches="tight")
plt.show()

# --- TABLE FIGURE (separate page) — INVERTED (classes as rows, species as columns) ---
tbl_df = class_by_species.T  # invert

fig_w = max(12, 2 + 0.35 * tbl_df.shape[1])   # width scales with #species (columns)
fig_h = max(6,  1.2 + 0.28 * tbl_df.shape[0]) # height scales with #classes (rows)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis('off')

tbl = ax.table(
    cellText=tbl_df.astype(int).values,   # remove .astype(int) if you want raw values
    rowLabels=tbl_df.index,
    colLabels=tbl_df.columns,
    cellLoc='center',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.25)  # widen cells / increase row height a bit

plt.tight_layout()
plt.savefig("SecretionUtilizationTable1stIterUnion.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

# --- FILES ---
summary_path = "summaryAll2ndIteration.xlsx"      # species *_summary sheets; KEGG IDs in 'products'
class_path   = "CompoundClassification.xlsx"       # use 'Graphing' sheet (KEGG ID, Name, Classification)

# --- COLOR PALETTE (unchanged) ---
color_palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b",
    "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#393b79", "#8c6d31",
    "#e7ba52", "#e7969c", "#7b4173", "#a55194", "#843c39", "#ad494a",
    "#d6616b", "#e6550d", "#9c9ede", "#b5cf6b", "#6b6ecf", "#bd9e39",
    "#cedb9c", "#637939", "#8ca252", "#bcbddc", "#c7e9c0", "#e9a3c9"
]

# --- Load classification mapping from Graphing sheet (preserve casing) ---
graph_df = pd.read_excel(
    class_path, sheet_name="Graphing", usecols=["KEGG ID", "Name", "Classification"]
).dropna(subset=["KEGG ID", "Classification"])
graph_df["KEGG ID"] = graph_df["KEGG ID"].astype(str).str.strip()
graph_df = graph_df.drop_duplicates(subset=["KEGG ID"], keep="first")

# Build class list for colors (from mapping)
all_unique_classes = sorted(graph_df["Classification"].unique())
class_color_mapping = {cls: color_palette[i % len(color_palette)] for i, cls in enumerate(all_unique_classes)}

# --- Read summary workbook ---
excel_data = pd.ExcelFile(summary_path)
summary_sheet_names = [s for s in excel_data.sheet_names if s.endswith("_summary")]

# Helper to pick KEGG-ID column (prefers 'products')
def pick_id_col(df, hint="products"):
    if hint in df.columns:
        return hint
    for c in df.columns:
        if str(c).strip().lower() == hint.lower():
            return c
    # heuristic: KEGG-like IDs
    for c in df.columns:
        s = df[c].dropna().astype(str)
        if len(s) and (s.str.match(r"^C\d{5}$").mean() > 0.5):
            return c
    return None

# Parse each species sheet → count KEGG IDs
all_substrate_frequencies = {}
for sheet in summary_sheet_names:
    species_name = sheet.replace("_summary", "")
    sheet_data = excel_data.parse(sheet)
    id_col = pick_id_col(sheet_data, "products")
    if id_col is None:
        continue
    substrate_counts = sheet_data[id_col].dropna().astype(str).str.strip().value_counts()
    all_substrate_frequencies[species_name] = pd.DataFrame({
        "Substrate": substrate_counts.index,
        "Frequency": substrate_counts.values
    })

# Combine species into one DataFrame
combined_substrate_frequencies = pd.concat(all_substrate_frequencies, names=["Species"]).reset_index(level=0)

# Merge with mapping to add Class (exact casing from Excel)
combined_substrate_with_class = combined_substrate_frequencies.merge(
    graph_df[["KEGG ID", "Classification"]],
    left_on="Substrate",
    right_on="KEGG ID",
    how="left"
)

# Drop unmapped and (optionally) exclude 'Free Radicals' while keeping other casings intact
combined_substrate_with_class = combined_substrate_with_class.dropna(subset=["Classification"])
combined_substrate_with_class = combined_substrate_with_class[
    combined_substrate_with_class["Classification"].astype(str).str.strip().str.lower() != "free radicals"
]

# Pivot: species × class → summed frequencies
class_by_species = combined_substrate_with_class.pivot_table(
    index="Species", columns="Classification", values="Frequency", aggfunc="sum", fill_value=0
)

# Sort species by total frequency (desc)
class_by_species["Total Frequency"] = class_by_species.sum(axis=1)
class_by_species = class_by_species.sort_values(by="Total Frequency", ascending=False).drop(columns="Total Frequency")

# Colors only for classes present in the pivot
colors = [class_color_mapping.get(cls, "#cccccc") for cls in class_by_species.columns]

# --- Plot stacked bar chart ---
plt.figure(figsize=(14, 8))
class_by_species.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 8),
    width=0.8,
    color=colors
)

plt.title("Substrate Class Utilization across Species")
plt.xlabel("Species")
plt.ylabel("Frequency of Utilization")

# Legend using exact class names from the data order
handles = [plt.Line2D([0], [0], color=class_color_mapping.get(cls, "#cccccc"), lw=4)
           for cls in class_by_species.columns]
plt.legend(handles, class_by_species.columns, title="Substrate Class",
           bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.savefig("SecretionUtilization2ndIterUnion.pdf", format="pdf", bbox_inches="tight")
plt.show()

# --- TABLE FIGURE (separate page) — INVERTED (classes as rows, species as columns) ---
tbl_df = class_by_species.T  # invert

fig_w = max(12, 2 + 0.35 * tbl_df.shape[1])   # width scales with #species (columns)
fig_h = max(6,  1.2 + 0.28 * tbl_df.shape[0]) # height scales with #classes (rows)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis('off')

tbl = ax.table(
    cellText=tbl_df.astype(int).values,   # remove .astype(int) if you want raw values
    rowLabels=tbl_df.index,
    colLabels=tbl_df.columns,
    cellLoc='center',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1, 1.25)  # widen cells / increase row height a bit

plt.tight_layout()
plt.savefig("SecretionUtilizationTable2ndIterUnion.pdf", format="pdf", bbox_inches="tight")
plt.show()
